# Web Analytics — Marketing Campaign Analysis

**Objectif :** Analyser les performances de 200 000 campagnes marketing digitales — CTR, CVR, CPA, ROI — jusqu'à l'étape EDA.  
**Dataset :** [Marketing Campaign Performance — Kaggle](https://www.kaggle.com/datasets/manishabhatt22/marketing-campaign-performance-dataset)  
**Stack :** Python · Pandas · NumPy · Matplotlib · Seaborn  

---

## Sommaire
1. Chargement et aperçu du dataset
2. Nettoyage des données
3. Analyse exploratoire (EDA)
   - 3.1 Distributions des KPIs
   - 3.2 CVR & CTR par canal
   - 3.3 ROI par type de campagne
   - 3.4 CPA par segment client
   - 3.5 Corrélations
   - 3.6 Insights & Recommandations

## 1. Chargement et aperçu du dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f8',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

df = pd.read_csv('data/marketing_campaign_sample.csv')
print(f'Shape : {df.shape}')
df.head()

In [ ]:
print('=== Types & valeurs manquantes ===')
info = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isnull().sum(),
    'missing_%': (df.isnull().sum() / len(df) * 100).round(1),
    'unique': df.nunique()
})
print(info.to_string())

## 2. Nettoyage des données

Deux opérations nécessaires avant l'analyse :
- `Acquisition_Cost` est au format string `"$1,234.56"` → conversion en float
- `CTR` n'est pas dans le dataset → calcul `Clicks / Impressions`

In [ ]:
# Nettoyage Acquisition_Cost
df['Acquisition_Cost'] = (
    df['Acquisition_Cost']
    .str.replace(r'[$,]', '', regex=True)
    .astype(float)
)

# Calcul CTR
df['CTR'] = df['Clicks'] / df['Impressions']

print('Vérification post-nettoyage :')
print(df[['Acquisition_Cost', 'CTR', 'Conversion_Rate', 'ROI']].describe().round(3))

## 3. Analyse Exploratoire (EDA)

### 3.1 Distributions des KPIs principaux

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Distribution des KPIs principaux', fontsize=14, fontweight='bold', y=1.01)

kpis = [
    ('Conversion_Rate', 'CVR', '#4285f4'),
    ('CTR',             'CTR', '#34a853'),
    ('ROI',             'ROI', '#fbbc04'),
    ('Acquisition_Cost','CPA ($)', '#ea4335'),
]

for ax, (col, label, color) in zip(axes.flat, kpis):
    ax.hist(df[col], bins=40, color=color, alpha=0.8, edgecolor='white', linewidth=0.4)
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1, label=f'Moyenne : {df[col].mean():.3f}')
    ax.axvline(df[col].median(), color='gray', linestyle=':', linewidth=1, label=f'Médiane : {df[col].median():.3f}')
    ax.set_title(label, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylabel('Fréquence')

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Statistiques descriptives :')
print(df[['Conversion_Rate','CTR','ROI','Acquisition_Cost']].describe().round(4))

### 3.2 CVR & CTR par canal

In [ ]:
cvr_ch = df.groupby('Channel_Used')['Conversion_Rate'].mean().sort_values(ascending=False)
ctr_ch = df.groupby('Channel_Used')['CTR'].mean().sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Performance par Canal', fontsize=13, fontweight='bold')

colors = ['#4285f4','#34a853','#fbbc04','#ea4335','#9c27b0','#00bcd4']

bars1 = ax1.barh(cvr_ch.index, cvr_ch.values * 100, color=colors, height=0.55)
for bar, val in zip(bars1, cvr_ch.values):
    ax1.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
             f'{val:.1%}', va='center', fontsize=10)
ax1.set_xlabel('CVR moyen (%)')
ax1.set_title('Conversion Rate par canal', fontweight='bold')
ax1.set_xlim(0, cvr_ch.max()*100 * 1.2)

bars2 = ax2.barh(ctr_ch.index, ctr_ch.values * 100, color=colors, height=0.55)
for bar, val in zip(bars2, ctr_ch.values):
    ax2.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val:.1%}', va='center', fontsize=10)
ax2.set_xlabel('CTR moyen (%)')
ax2.set_title('Click-Through Rate par canal', fontweight='bold')
ax2.set_xlim(0, ctr_ch.max()*100 * 1.2)

plt.tight_layout()
plt.savefig('eda_canal_cvr_ctr.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nTableau comparatif CVR vs CTR par canal :')
print(pd.DataFrame({'CVR': cvr_ch.round(4), 'CTR': ctr_ch.round(4)}).to_string())

> **Insight :** Google Ads a le CVR le plus élevé (8.17%) mais le CTR le plus faible (13.2%) — ce canal génère moins de clics mais une audience plus qualifiée. À l'inverse, Facebook génère le plus de clics (CTR 15%) mais convertit moins bien. Recommandation : prioriser Google Ads pour les campagnes de conversion, Facebook pour la notoriété.

### 3.3 ROI par type de campagne

In [ ]:
roi_type = df.groupby('Campaign_Type')['ROI'].agg(['mean','std','count']).sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(roi_type.index, roi_type['mean'],
              color=['#4285f4','#34a853','#fbbc04','#ea4335','#9c27b0'],
              width=0.55,
              yerr=roi_type['std'],
              capsize=4,
              error_kw={'linewidth':1, 'alpha':0.6})

for bar, (_, row) in zip(bars, roi_type.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.12,
            f'{row["mean"]:.2f}x', ha='center', fontsize=10, fontweight='bold')

ax.axhline(df['ROI'].mean(), color='black', linestyle='--', linewidth=1,
           label=f'ROI moyen global : {df["ROI"].mean():.2f}x')
ax.set_ylabel('ROI moyen')
ax.set_title('ROI moyen par type de campagne (avec écart-type)', fontweight='bold')
ax.legend()
ax.tick_params(axis='x', rotation=10)
plt.tight_layout()
plt.savefig('eda_roi_type.png', dpi=120, bbox_inches='tight')
plt.show()

print(roi_type.round(3))

> **Insight :** Les campagnes Social Media et Influencer ont le ROI le plus élevé (~5.1x). Les écarts-types sont similaires entre types — la variabilité de performance ne dépend pas du type de campagne mais d'autres facteurs (canal, audience, durée).

### 3.4 CPA par segment client

In [ ]:
cpa_seg = df.groupby('Customer_Segment')['Acquisition_Cost'].mean().sort_values()
mean_cpa = df['Acquisition_Cost'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
colors_bar = ['#34a853' if v <= mean_cpa else '#ea4335' for v in cpa_seg.values]
bars = ax.barh(cpa_seg.index, cpa_seg.values, color=colors_bar, height=0.5, alpha=0.85)

for bar, val in zip(bars, cpa_seg.values):
    ax.text(bar.get_width() + 80, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=10)

ax.axvline(mean_cpa, color='black', linestyle='--', linewidth=1,
           label=f'CPA moyen : ${mean_cpa:,.0f}')
ax.set_xlabel('CPA moyen ($)')
ax.set_title('Coût par Acquisition moyen par Segment client', fontweight='bold')
ax.legend()
ax.set_xlim(0, cpa_seg.max() * 1.15)
plt.tight_layout()
plt.savefig('eda_cpa_segment.png', dpi=120, bbox_inches='tight')
plt.show()

print(cpa_seg.round(0).to_string())

> **Insight :** Les CPA par segment sont très proches ($12,472 à $12,552) — la segmentation client n'explique pas les variations de coût d'acquisition. D'autres variables (canal, durée, type) ont plus d'impact sur le CPA.

### 3.5 Matrice de corrélation

In [ ]:
numeric_cols = ['CTR', 'Conversion_Rate', 'Acquisition_Cost', 'ROI', 'Engagement_Score', 'Clicks', 'Impressions']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.8})
ax.set_title('Matrice de corrélation — KPIs numériques', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('eda_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

### 3.6 Insights & Recommandations

| # | Insight | Recommandation |
|---|---------|----------------|
| 1 | Google Ads : CVR max (8.17%) mais CTR min (13.2%) | Prioriser Google Ads pour les campagnes de **conversion** |
| 2 | Facebook : CTR max (15%) mais CVR faible | Utiliser Facebook pour la **notoriété** et le haut de funnel |
| 3 | Social Media & Influencer : ROI le plus élevé (~5.1x) | Augmenter le budget sur ces types de campagne |
| 4 | CPA quasi-identique entre segments ($12,472–$12,552) | Le canal et le type ont plus d'impact sur le CPA que le segment |
| 5 | CVR distribué de 1% à 15% (médiane 8%) | Fort potentiel d'optimisation sur les campagnes sous-performantes |

---

**Prochaines étapes (hors scope EDA) :**
- Modélisation prédictive du CVR (régression, XGBoost)
- Segmentation des campagnes sous-performantes
- Analyse temporelle (évolution 2021)
- Dashboard interactif Looker Studio